# Section 0 — Configuration
All hyperparameters and global settings for the pipeline.  
Change `DEBUG = False` and `SUBTASK` as needed for full training or different subtasks.

In [1]:
# ========================== SECTION 0: CONFIGURATION ==========================
DEBUG = True              # Set False for full training
SUBTASK = "A"             # "A", "B", or "C"
BATCH_SIZE = 32
HIDDEN_DIM = 128
NUM_EPOCHS = 20
LEARNING_RATE = 1e-3
DROPOUT = 0.2
MAX_NODES = 512           # Truncate graphs larger than this
CACHE_DIR = "./graph_cache"
PATIENCE = 5              # Early stopping patience
WEIGHT_DECAY = 1e-4
DEBUG_SAMPLE_SIZE = 1000  # Rows to sample in debug mode
RANDOM_SEED = 42

# Subtask → number of classes mapping
NUM_CLASSES_MAP = {"A": 2, "B": 11, "C": 4}
NUM_CLASSES = NUM_CLASSES_MAP[SUBTASK]

print(f"Configuration loaded: SUBTASK={SUBTASK}, DEBUG={DEBUG}, "
      f"NUM_CLASSES={NUM_CLASSES}, DEVICE=pending")

Configuration loaded: SUBTASK=A, DEBUG=True, NUM_CLASSES=2, DEVICE=pending


# Section 1 — Install Dependencies
Conditionally install only missing packages using `importlib`.  
Handles `torch-geometric` installation with correct `--find-links` URL for the detected PyTorch version.

In [ ]:
# ========================== SECTION 1: INSTALL DEPENDENCIES ==========================
import subprocess, sys, importlib.util

def install_if_missing(package, pip_name=None):
    """Install a package if not already available.
    Args:
        package: The import name to check.
        pip_name: The pip package name (if different from import name).
    """
    if importlib.util.find_spec(package) is None:
        name = pip_name or package
        print(f"Installing {name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", name])
    else:
        print(f"✓ {package} already installed")

# Core scientific packages
for pkg in ["numpy", "pandas", "sklearn", "tqdm", "matplotlib"]:
    pip_name = "scikit-learn" if pkg == "sklearn" else None
    install_if_missing(pkg, pip_name)

# HuggingFace datasets
install_if_missing("datasets")

# PyTorch (skip if already available)
install_if_missing("torch")

# Tree-sitter core + language grammars
install_if_missing("tree_sitter", "tree-sitter")
ts_langs = {
    "tree_sitter_python": "tree-sitter-python",
    "tree_sitter_cpp": "tree-sitter-cpp",
    "tree_sitter_java": "tree-sitter-java",
    "tree_sitter_javascript": "tree-sitter-javascript",
    "tree_sitter_go": "tree-sitter-go",
    "tree_sitter_c_sharp": "tree-sitter-c-sharp",
    "tree_sitter_c": "tree-sitter-c",
}
for imp_name, pip_name in ts_langs.items():
    install_if_missing(imp_name, pip_name)

# torch-geometric (needs special --find-links handling)
if importlib.util.find_spec("torch_geometric") is None:
    import torch
    tv = torch.__version__.split("+")[0]  # e.g. "2.2.0"
    cuda_tag = "cu" + torch.version.cuda.replace(".", "") if torch.cuda.is_available() and torch.version.cuda else "cpu"
    find_links = f"https://data.pyg.org/whl/torch-{tv}+{cuda_tag}.html"
    print(f"Installing torch-geometric (torch={tv}, backend={cuda_tag})...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "torch-geometric", "--find-links", find_links])
else:
    print("✓ torch_geometric already installed")

# Verify critical imports
import torch, torch_geometric
print(f"\n✓ All dependencies installed. torch={torch.__version__}, "
      f"torch_geometric={torch_geometric.__version__}")

# Section 2 — Imports & Setup
Import all libraries, set random seeds for reproducibility, detect compute device, and print system information.

In [ ]:
# ========================== SECTION 2: IMPORTS & SETUP ==========================
import os, sys, json, random, time, warnings, hashlib, gc, pathlib
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam

import torch_geometric
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool

from sklearn.metrics import (f1_score, precision_score, recall_score,
                             accuracy_score, confusion_matrix, classification_report)
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# Tree-sitter imports
from tree_sitter import Language, Parser
import tree_sitter_python as ts_python
import tree_sitter_cpp as ts_cpp
import tree_sitter_java as ts_java
import tree_sitter_javascript as ts_javascript
import tree_sitter_go as ts_go
import tree_sitter_c_sharp as ts_c_sharp
import tree_sitter_c as ts_c

warnings.filterwarnings("ignore")

# ---------- Reproducibility ----------
def seed_everything(seed):
    """Set all random seeds for reproducibility.
    Args:
        seed: Integer seed value.
    """
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(RANDOM_SEED)

# ---------- Device detection ----------
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# ---------- System info ----------
import platform
print(f"Python        : {platform.python_version()}")
print(f"PyTorch       : {torch.__version__}")
print(f"PyG           : {torch_geometric.__version__}")
print(f"Device        : {device}")
print(f"OS            : {platform.system()} {platform.machine()}")
try:
    import psutil
    ram_gb = psutil.virtual_memory().total / (1024**3)
    print(f"RAM           : {ram_gb:.1f} GB")
except ImportError:
    pass
if device.type == "cuda":
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
    print(f"VRAM          : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# Section 3 — Dataset Loading
Load the SemEval-2026 Task 13 dataset with a cascading strategy:
1. **Kaggle path** (fastest, no download)
2. **HuggingFace Hub** (auto-download)
3. **Local parquet files** (manual download)

Also loads label mappings, cleans the data, and optionally samples for debug mode.

In [ ]:
# ========================== SECTION 3: DATASET LOADING ==========================

def load_dataset_for_subtask(subtask="A"):
    """Load train/val datasets for the given subtask using a cascade of sources.

    Args:
        subtask: One of 'A', 'B', or 'C'.

    Returns:
        Tuple of (train_df, val_df, label_to_id, id_to_label).

    Raises:
        FileNotFoundError: If no data source is available.
    """
    task_key = f"task_{subtask.lower()}"

    # --- Strategy 1: Kaggle path ---
    kaggle_base = f"/kaggle/input/semeval-2026-task13/{task_key}"
    if os.path.isdir(kaggle_base):
        print(f"Loading from Kaggle: {kaggle_base}")
        train_files = sorted(pathlib.Path(kaggle_base).glob("*training*"))
        val_files = sorted(pathlib.Path(kaggle_base).glob("*validation*"))
        train_df = pd.concat([pd.read_parquet(f) for f in train_files], ignore_index=True)
        val_df = pd.read_parquet(val_files[0]) if val_files else None
        # Load label mappings
        label_to_id, id_to_label = _load_label_maps(kaggle_base)
        return train_df, val_df, label_to_id, id_to_label

    # --- Strategy 2: HuggingFace Hub ---
    try:
        from datasets import load_dataset as hf_load
        print(f"Loading from HuggingFace: DaniilOr/SemEval-2026-Task13 ({task_key})")
        ds = hf_load("DaniilOr/SemEval-2026-Task13", task_key)
        train_df = ds["train"].to_pandas()
        val_df = ds["validation"].to_pandas() if "validation" in ds else None
        # Build label maps from the data
        unique_labels = sorted(train_df["label"].unique())
        id_to_label = {i: str(i) for i in unique_labels}
        label_to_id = {str(i): i for i in unique_labels}
        return train_df, val_df, label_to_id, id_to_label
    except Exception as e:
        print(f"HuggingFace failed: {e}")

    # --- Strategy 3: Local parquet files ---
    search_dirs = [
        f"./{task_key}", f"./SemEval-2026-Task13/{task_key}",
        f"./SemEval-2026-Task13/SemEval-2026-Task13/{task_key}",
        f"../SemEval-2026-Task13/{task_key}",
    ]
    for base in search_dirs:
        if os.path.isdir(base):
            print(f"Loading from local: {base}")
            train_files = sorted(pathlib.Path(base).glob("*training*"))
            val_files = sorted(pathlib.Path(base).glob("*validation*"))
            if train_files:
                train_df = pd.concat([pd.read_parquet(f) for f in train_files], ignore_index=True)
                val_df = pd.read_parquet(val_files[0]) if val_files else None
                label_to_id, id_to_label = _load_label_maps(base)
                return train_df, val_df, label_to_id, id_to_label

    raise FileNotFoundError(
        f"Could not find dataset for {task_key}. Please place parquet files in ./{task_key}/ "
        "or ensure HuggingFace access.")


def _load_label_maps(base_dir):
    """Load label_to_id.json and id_to_label.json from a directory.

    Args:
        base_dir: Directory containing JSON mapping files.

    Returns:
        Tuple of (label_to_id dict, id_to_label dict).
    """
    l2i_path = os.path.join(base_dir, "label_to_id.json")
    i2l_path = os.path.join(base_dir, "id_to_label.json")
    if os.path.exists(l2i_path):
        with open(l2i_path) as f:
            label_to_id = json.load(f)
        with open(i2l_path) as f:
            id_to_label = json.load(f)
        # Ensure keys are correct types
        id_to_label = {int(k): v for k, v in id_to_label.items()}
    else:
        label_to_id, id_to_label = {}, {}
    return label_to_id, id_to_label


# ---------- Load the data ----------
train_df, val_df, label_to_id, id_to_label = load_dataset_for_subtask(SUBTASK)

# ---------- Clean data ----------
for df_name, df in [("train", train_df), ("val", val_df)]:
    if df is None:
        continue
    before = len(df)
    mask = df["code"].notna() & (df["code"].str.strip() != "")
    if df_name == "train":
        train_df = train_df[mask].reset_index(drop=True)
    else:
        val_df = val_df[mask].reset_index(drop=True)
    print(f"{df_name}: dropped {before - mask.sum()} null/empty rows")

# ---------- Debug sampling ----------
if DEBUG:
    print(f"\nDEBUG mode: sampling {DEBUG_SAMPLE_SIZE} rows (stratified)...")
    train_df, _ = train_test_split(
        train_df, train_size=min(DEBUG_SAMPLE_SIZE, len(train_df)),
        stratify=train_df["label"], random_state=RANDOM_SEED
    )
    train_df = train_df.reset_index(drop=True)
    if val_df is not None and len(val_df) > DEBUG_SAMPLE_SIZE:
        val_df, _ = train_test_split(
            val_df, train_size=min(DEBUG_SAMPLE_SIZE // 2, len(val_df)),
            stratify=val_df["label"], random_state=RANDOM_SEED
        )
        val_df = val_df.reset_index(drop=True)

# ---------- Print stats ----------
print(f"\n{'='*50}")
print(f"Subtask {SUBTASK} — Dataset Summary")
print(f"{'='*50}")
print(f"Train samples: {len(train_df):,}")
if val_df is not None:
    print(f"Val samples  : {len(val_df):,}")
print(f"\nClass distribution (train):")
print(train_df["label"].value_counts().to_string())
if "language" in train_df.columns:
    print(f"\nLanguage distribution (train):")
    print(train_df["language"].value_counts().to_string())
code_lens = train_df["code"].str.len()
print(f"\nCode length — mean: {code_lens.mean():.0f}, "
      f"median: {code_lens.median():.0f}, max: {code_lens.max():,}")

# Section 4 — AST Parser Setup
Initialize Tree-sitter parsers for all supported languages using the modern API (no manual `.so` compilation).  
Languages: Python, C++, Java, JavaScript, Go, C#, C.

Includes a `parse_code()` function with error handling and a **smoke test** cell.

In [ ]:
# ========================== SECTION 4: AST PARSER SETUP ==========================

def build_parsers():
    """Build a dictionary of Tree-sitter parsers keyed by language name.

    Returns:
        Dict mapping language name (str) to a configured tree_sitter.Parser.
    """
    lang_modules = {
        "python": ts_python,
        "cpp": ts_cpp, "c++": ts_cpp,
        "java": ts_java,
        "javascript": ts_javascript, "js": ts_javascript,
        "go": ts_go,
        "c_sharp": ts_c_sharp, "c#": ts_c_sharp, "csharp": ts_c_sharp,
        "c": ts_c,
    }
    parsers = {}
    for name, mod in lang_modules.items():
        lang = Language(mod.language())
        p = Parser(lang)
        parsers[name] = p
    print(f"Initialized {len(parsers)} parser entries for "
          f"{len(set(id(v) for v in parsers.values()))} unique languages")
    return parsers

parsers = build_parsers()


def normalize_language(lang_str):
    """Normalize a language string to match parser dict keys.

    Args:
        lang_str: Raw language string from the dataset.

    Returns:
        Normalized language key string, or None if unsupported.
    """
    if lang_str is None:
        return None
    s = lang_str.strip().lower()
    mapping = {
        "python": "python", "py": "python",
        "c++": "cpp", "cpp": "cpp",
        "java": "java",
        "javascript": "javascript", "js": "javascript",
        "go": "go", "golang": "go",
        "c#": "c_sharp", "csharp": "c_sharp", "c_sharp": "c_sharp",
        "c": "c",
    }
    return mapping.get(s)


def parse_code(code_str, language, parsers_dict):
    """Parse source code into a Tree-sitter AST.

    Args:
        code_str: Source code as a string.
        language: Language name string.
        parsers_dict: Dict of parsers from build_parsers().

    Returns:
        tree_sitter.Node (root node) or None if parsing fails.
    """
    try:
        lang_key = normalize_language(language)
        if lang_key is None or lang_key not in parsers_dict:
            return None
        parser = parsers_dict[lang_key]
        code_bytes = code_str.encode("utf-8") if isinstance(code_str, str) else code_str
        tree = parser.parse(code_bytes)
        if tree and tree.root_node and tree.root_node.child_count > 0:
            return tree.root_node
        return None
    except Exception:
        return None


def ast_depth(node, current_depth=0):
    """Compute the depth of an AST subtree.

    Args:
        node: A tree_sitter.Node.
        current_depth: Current recursion depth.

    Returns:
        Integer depth of the deepest node.
    """
    if node.child_count == 0:
        return current_depth
    return max(ast_depth(c, current_depth + 1) for c in node.children)


def ast_node_count(node):
    """Count total nodes in an AST subtree.

    Args:
        node: A tree_sitter.Node.

    Returns:
        Integer count of all nodes (including the given node).
    """
    return 1 + sum(ast_node_count(c) for c in node.children)

### Smoke Test — AST Parsing
Parse one example per language from the training data and verify the AST is valid.

In [ ]:
# ---------- SMOKE TEST: AST Parsing ----------
print("AST Parsing Smoke Test")
print("-" * 60)

if "language" in train_df.columns:
    test_langs = train_df["language"].dropna().unique()
else:
    test_langs = ["python"]

for lang in test_langs:
    if "language" in train_df.columns:
        subset = train_df[train_df["language"] == lang]
    else:
        subset = train_df
    if len(subset) == 0:
        continue
    sample_code = subset.iloc[0]["code"]
    root = parse_code(sample_code, lang, parsers)
    if root is not None:
        depth = ast_depth(root)
        count = ast_node_count(root)
        print(f"✓ {lang:12s} → depth={depth:3d}, nodes={count:5d}")
        assert depth > 0, f"AST depth should be > 0 for {lang}"
        assert count > 1, f"AST node count should be > 1 for {lang}"
    else:
        nl = normalize_language(lang)
        if nl and nl in parsers:
            print(f"✗ {lang:12s} → parsing failed (sample may be malformed)")
        else:
            print(f"✗ {lang:12s} → unsupported language (no parser)")

print("\n✓ AST parser smoke test complete")

# Section 5 — AST to Graph Conversion
Convert parsed ASTs into graph representations compatible with PyTorch Geometric.

**Node features** (5-dimensional):
1. Node type index (from a vocabulary built on training data)
2. Normalized tree depth
3. Normalized child count
4. Is-leaf flag (0/1)
5. Is-named flag (0/1 — Tree-sitter distinguishes named vs anonymous nodes)

**Edges**: Bidirectional parent↔child edges.  
**Truncation**: Graphs larger than `MAX_NODES` are truncated during BFS.

In [ ]:
# ========================== SECTION 5: AST TO GRAPH CONVERSION ==========================

def build_node_vocab(df, parsers_dict, max_samples=None):
    """Build a vocabulary of AST node types from the training data.

    Args:
        df: Training DataFrame with 'code' and 'language' columns.
        parsers_dict: Dict of Tree-sitter parsers.
        max_samples: Optional max number of samples to scan.

    Returns:
        Dict mapping node type string to integer index.
    """
    node_types = set()
    samples = df if max_samples is None else df.head(max_samples)
    count = 0
    for _, row in tqdm(samples.iterrows(), total=len(samples), desc="Building node vocab"):
        lang = row.get("language", "python")
        root = parse_code(row["code"], lang, parsers_dict)
        if root is None:
            continue
        # BFS to collect node types
        queue = [root]
        while queue:
            n = queue.pop(0)
            node_types.add(n.type)
            queue.extend(n.children)
        count += 1
    # Sort for deterministic ordering; reserve 0 for <UNK>
    sorted_types = sorted(node_types)
    vocab = {"<UNK>": 0}
    for i, t in enumerate(sorted_types, start=1):
        vocab[t] = i
    print(f"Node vocab built: {len(vocab)} types from {count} samples")
    return vocab


def ast_to_graph(root_node, node_vocab, max_nodes=512):
    """Convert an AST root node to a graph (node features + edge index).

    Args:
        root_node: tree_sitter.Node root of the AST.
        node_vocab: Dict mapping node type str to int index.
        max_nodes: Maximum number of nodes to include.

    Returns:
        Tuple of (node_features: Tensor[N,5], edge_index: Tensor[2,E])
        or None if the graph has 0 nodes.

    Raises:
        None — returns None on degenerate graphs.
    """
    # BFS traversal
    nodes = []        # list of (node, parent_idx, depth)
    queue = [(root_node, -1, 0)]
    edges_src, edges_dst = [], []

    while queue and len(nodes) < max_nodes:
        node, parent_idx, depth = queue.pop(0)
        cur_idx = len(nodes)
        nodes.append((node, depth))

        # Add bidirectional edge to parent
        if parent_idx >= 0:
            edges_src.append(parent_idx)
            edges_dst.append(cur_idx)
            edges_src.append(cur_idx)
            edges_dst.append(parent_idx)

        # Enqueue children
        for child in node.children:
            if len(nodes) + len(queue) < max_nodes:
                queue.append((child, cur_idx, depth + 1))

    if len(nodes) == 0:
        return None

    # Compute normalization constants
    max_depth = max(d for _, d in nodes) if nodes else 1
    max_depth = max(max_depth, 1)
    max_children = max(n.child_count for n, _ in nodes) if nodes else 1
    max_children = max(max_children, 1)

    # Build feature matrix [N, 5]
    features = []
    for node, depth in nodes:
        type_idx = node_vocab.get(node.type, node_vocab["<UNK>"])
        feat = [
            type_idx / max(len(node_vocab), 1),  # normalized type index
            depth / max_depth,                     # normalized depth
            node.child_count / max_children,       # normalized child count
            1.0 if node.child_count == 0 else 0.0, # is_leaf
            1.0 if node.is_named else 0.0,         # is_named
        ]
        features.append(feat)

    node_features = torch.tensor(features, dtype=torch.float)

    if len(edges_src) > 0:
        edge_index = torch.tensor([edges_src, edges_dst], dtype=torch.long)
    else:
        # Self-loop for single-node graphs
        edge_index = torch.tensor([[0], [0]], dtype=torch.long)

    return node_features, edge_index


# ---------- Build vocabulary from training data ----------
vocab_sample_size = len(train_df) if not DEBUG else min(500, len(train_df))
node_vocab = build_node_vocab(train_df, parsers, max_samples=vocab_sample_size)

### Smoke Test — AST to Graph

In [ ]:
# ---------- SMOKE TEST: AST → Graph ----------
print("AST → Graph Smoke Test")
print("-" * 60)

test_samples = train_df.head(5)
for i, (_, row) in enumerate(test_samples.iterrows()):
    lang = row.get("language", "python")
    root = parse_code(row["code"], lang, parsers)
    if root is None:
        print(f"Sample {i}: parsing failed — skipped")
        continue
    result = ast_to_graph(root, node_vocab, MAX_NODES)
    if result is None:
        print(f"Sample {i}: empty graph — skipped")
        continue
    nf, ei = result
    assert nf.ndim == 2 and nf.shape[1] == 5, f"Expected [N,5], got {nf.shape}"
    assert ei.ndim == 2 and ei.shape[0] == 2, f"Expected [2,E], got {ei.shape}"
    assert nf.shape[0] <= MAX_NODES, f"Nodes {nf.shape[0]} exceeds MAX_NODES"
    print(f"Sample {i} ({lang}): nodes={nf.shape[0]}, edges={ei.shape[1]}, "
          f"features={nf.shape[1]}")

print("\n✓ AST → Graph smoke test passed")

# Section 6 — Graph Dataset Class
Build a dataset class that converts code samples into PyG `Data` objects with disk caching.  
Uses a list-based approach with per-sample `.pt` file caching for memory efficiency.  
Rows where AST parsing fails are skipped and logged.

In [ ]:
# ========================== SECTION 6: GRAPH DATASET CLASS ==========================

class CodeGraphDataset:
    """Dataset that converts code samples to cached PyG Data objects.

    Args:
        df: DataFrame with 'code', 'label', and optionally 'language' columns.
        parsers_dict: Dict of Tree-sitter parsers.
        node_vocab: Dict mapping node type string to integer index.
        cache_dir: Directory to cache processed .pt files.
        max_nodes: Maximum nodes per graph.
        split_name: Name used for cache subdirectory (e.g. 'train', 'val').
    """

    def __init__(self, df, parsers_dict, node_vocab, cache_dir, max_nodes=512,
                 split_name="data"):
        self.df = df.reset_index(drop=True)
        self.parsers = parsers_dict
        self.node_vocab = node_vocab
        self.max_nodes = max_nodes
        self.cache_path = os.path.join(cache_dir, split_name)
        os.makedirs(self.cache_path, exist_ok=True)

        self.valid_indices = []
        self.skip_count = 0
        self._process()

    def _get_cache_file(self, idx):
        """Return the cache file path for a given index.

        Args:
            idx: Integer index into the DataFrame.

        Returns:
            String path to the .pt cache file.
        """
        return os.path.join(self.cache_path, f"graph_{idx}.pt")

    def _process(self):
        """Convert all samples to graphs and cache to disk."""
        print(f"Processing {len(self.df)} samples → {self.cache_path}")
        for idx in tqdm(range(len(self.df)), desc="Building graphs"):
            cache_file = self._get_cache_file(idx)

            # Use cache if available
            if os.path.exists(cache_file):
                self.valid_indices.append(idx)
                continue

            row = self.df.iloc[idx]
            lang = row.get("language", "python")
            root = parse_code(row["code"], lang, self.parsers)

            if root is None:
                self.skip_count += 1
                continue

            result = ast_to_graph(root, self.node_vocab, self.max_nodes)
            if result is None or result[0].shape[0] == 0:
                self.skip_count += 1
                continue

            node_features, edge_index = result
            data = Data(
                x=node_features,
                edge_index=edge_index,
                y=torch.tensor(row["label"], dtype=torch.long),
                num_nodes=node_features.shape[0],
            )
            torch.save(data, cache_file)
            self.valid_indices.append(idx)

        print(f"✓ {len(self.valid_indices)} graphs cached, "
              f"{self.skip_count} skipped (parse failures)")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        real_idx = self.valid_indices[idx]
        return torch.load(self._get_cache_file(real_idx), weights_only=False)

    def to_list(self):
        """Return all Data objects as a list for PyG DataLoader.

        Returns:
            List of torch_geometric.data.Data objects.
        """
        return [self[i] for i in range(len(self))]


# ---------- Build datasets ----------
print("Building training graph dataset...")
train_dataset = CodeGraphDataset(
    train_df, parsers, node_vocab, CACHE_DIR,
    max_nodes=MAX_NODES, split_name=f"task_{SUBTASK.lower()}_train"
)

if val_df is not None:
    print("\nBuilding validation graph dataset...")
    val_dataset = CodeGraphDataset(
        val_df, parsers, node_vocab, CACHE_DIR,
        max_nodes=MAX_NODES, split_name=f"task_{SUBTASK.lower()}_val"
    )
else:
    val_dataset = None

print(f"\nTrain graphs: {len(train_dataset)}")
if val_dataset:
    print(f"Val graphs  : {len(val_dataset)}")

### Smoke Test — Graph Dataset

In [ ]:
# ---------- SMOKE TEST: Graph Dataset ----------
print("Graph Dataset Smoke Test")
print("-" * 60)

assert len(train_dataset) <= len(train_df), "Dataset size exceeds DataFrame"
assert len(train_dataset) > 0, "Dataset is empty — all samples failed parsing"

sample = train_dataset[0]
assert hasattr(sample, "x"), "Data missing 'x' (node features)"
assert hasattr(sample, "edge_index"), "Data missing 'edge_index'"
assert hasattr(sample, "y"), "Data missing 'y' (label)"
assert sample.x.ndim == 2 and sample.x.shape[1] == 5, f"Unexpected feature shape: {sample.x.shape}"
print(f"Sample 0: nodes={sample.x.shape[0]}, edges={sample.edge_index.shape[1]}, "
      f"label={sample.y.item()}")

# Test DataLoader batching
loader_test = PyGDataLoader(train_dataset.to_list()[:8], batch_size=4, shuffle=False)
batch = next(iter(loader_test))
print(f"Batch: x={batch.x.shape}, edge_index={batch.edge_index.shape}, "
      f"y={batch.y.shape}, batch_vec={batch.batch.shape}")

print("\n✓ Graph Dataset smoke test passed")

# Section 7 — GNN Model
Architecture:
```
Linear(feat_dim → hidden) → GCNConv → GCNConv → GATConv(heads=4) → global_mean_pool → MLP → logits
```
No softmax in `forward()` — `CrossEntropyLoss` handles it externally.

In [ ]:
# ========================== SECTION 7: GNN MODEL ==========================

class CodeGNN(nn.Module):
    """Graph Neural Network for code classification.

    Architecture: Linear → 2×GCN → GAT(4 heads) → global_mean_pool → MLP.

    Args:
        feat_dim: Input feature dimension per node.
        hidden_dim: Hidden dimension for all layers.
        num_classes: Number of output classes.
        dropout: Dropout probability.
    """

    def __init__(self, feat_dim, hidden_dim, num_classes, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(feat_dim, hidden_dim)
        self.gcn1 = GCNConv(hidden_dim, hidden_dim)
        self.gcn2 = GCNConv(hidden_dim, hidden_dim)
        self.gat = GATConv(hidden_dim, hidden_dim, heads=4, concat=False)
        self.fc1 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.fc2 = nn.Linear(hidden_dim // 2, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index, batch):
        """Forward pass.

        Args:
            x: Node feature matrix [N, feat_dim].
            edge_index: Edge index [2, E].
            batch: Batch vector [N] mapping nodes to graphs.

        Returns:
            Logits tensor [B, num_classes].
        """
        # Input projection
        h = F.relu(self.input_proj(x))

        # GCN layers
        h = F.relu(self.gcn1(h, edge_index))
        h = self.dropout(h)
        h = F.relu(self.gcn2(h, edge_index))
        h = self.dropout(h)

        # GAT layer
        h = F.relu(self.gat(h, edge_index))

        # Global pooling
        h = global_mean_pool(h, batch)  # [B, hidden_dim]

        # MLP classifier
        h = F.relu(self.fc1(h))
        h = self.dropout(h)
        h = self.fc2(h)  # [B, num_classes] — raw logits, no softmax

        return h


# ---------- Instantiate model ----------
FEAT_DIM = 5  # node features: type_idx, depth, children, is_leaf, is_named
model = CodeGNN(
    feat_dim=FEAT_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=NUM_CLASSES,
    dropout=DROPOUT
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: CodeGNN")
print(f"  Parameters: {total_params:,} total, {trainable_params:,} trainable")
print(f"  Device: {device}")
print(model)

### Smoke Test — GNN Model

In [ ]:
# ---------- SMOKE TEST: GNN Model ----------
print("GNN Model Smoke Test")
print("-" * 60)

# Create a dummy batch from the first few samples
dummy_list = train_dataset.to_list()[:BATCH_SIZE]
dummy_loader = PyGDataLoader(dummy_list, batch_size=len(dummy_list))
dummy_batch = next(iter(dummy_loader)).to(device)

with torch.no_grad():
    out = model(dummy_batch.x, dummy_batch.edge_index, dummy_batch.batch)

expected_shape = (len(dummy_list), NUM_CLASSES)
assert out.shape == expected_shape, f"Expected {expected_shape}, got {out.shape}"
print(f"Input : x={dummy_batch.x.shape}, edge_index={dummy_batch.edge_index.shape}")
print(f"Output: {out.shape} (expected {expected_shape})")
print(f"Sample logits: {out[0].cpu().numpy()}")

print("\n✓ GNN model smoke test passed")

# Section 8 — Training Pipeline
- Adam optimizer with weight decay
- Class-weighted `CrossEntropyLoss` (critical for imbalanced Subtask B)
- Early stopping on validation macro F1 (patience=5)
- Best model checkpoint saved to `best_model.pt`

In [ ]:
# ========================== SECTION 8: TRAINING PIPELINE ==========================

def compute_class_weights(dataset):
    """Compute inverse-frequency class weights for CrossEntropyLoss.

    Args:
        dataset: CodeGraphDataset instance.

    Returns:
        Tensor of class weights on the compute device.
    """
    labels = [dataset[i].y.item() for i in range(len(dataset))]
    counts = Counter(labels)
    total = sum(counts.values())
    weights = []
    for c in range(NUM_CLASSES):
        cnt = counts.get(c, 1)
        weights.append(total / (NUM_CLASSES * cnt))
    w = torch.tensor(weights, dtype=torch.float)
    w = w / w.sum() * NUM_CLASSES  # normalize so mean weight = 1
    return w.to(device)


def train_epoch(model, loader, optimizer, criterion, dev):
    """Train for one epoch.

    Args:
        model: CodeGNN model.
        loader: PyG DataLoader.
        optimizer: torch.optim.Optimizer.
        criterion: Loss function.
        dev: torch.device.

    Returns:
        Dict with 'loss' (float) and 'f1' (float).
    """
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for batch in tqdm(loader, desc="  Train", leave=False):
        batch = batch.to(dev)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(out, batch.y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * batch.y.size(0)
        preds = out.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(batch.y.cpu().numpy())

    avg_loss = total_loss / len(all_labels)
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return {"loss": avg_loss, "f1": f1}


def eval_epoch(model, loader, criterion, dev):
    """Evaluate on a dataset.

    Args:
        model: CodeGNN model.
        loader: PyG DataLoader.
        criterion: Loss function.
        dev: torch.device.

    Returns:
        Dict with 'loss', 'f1', 'preds', 'labels'.
    """
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc="  Eval ", leave=False):
            batch = batch.to(dev)
            out = model(batch.x, batch.edge_index, batch.batch)
            loss = criterion(out, batch.y)
            total_loss += loss.item() * batch.y.size(0)
            preds = out.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(batch.y.cpu().numpy())

    avg_loss = total_loss / max(len(all_labels), 1)
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return {"loss": avg_loss, "f1": f1, "preds": all_preds, "labels": all_labels}


# ---------- Prepare DataLoaders ----------
train_list = train_dataset.to_list()
train_loader = PyGDataLoader(train_list, batch_size=BATCH_SIZE, shuffle=True)
if val_dataset and len(val_dataset) > 0:
    val_list = val_dataset.to_list()
    val_loader = PyGDataLoader(val_list, batch_size=BATCH_SIZE, shuffle=False)
else:
    val_loader = None

# ---------- Loss, optimizer ----------
class_weights = compute_class_weights(train_dataset)
print(f"Class weights: {class_weights.cpu().numpy()}")
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# ---------- Training loop ----------
best_val_f1 = 0.0
patience_counter = 0
history = {"train_loss": [], "val_loss": [], "train_f1": [], "val_f1": []}

print(f"\nStarting training for {NUM_EPOCHS} epochs...")
print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Val Loss':>10} | "
      f"{'Train F1':>9} | {'Val F1':>9} | {'Time':>6}")
print("-" * 65)

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.time()

    train_metrics = train_epoch(model, train_loader, optimizer, criterion, device)
    history["train_loss"].append(train_metrics["loss"])
    history["train_f1"].append(train_metrics["f1"])

    if val_loader:
        val_metrics = eval_epoch(model, val_loader, criterion, device)
        history["val_loss"].append(val_metrics["loss"])
        history["val_f1"].append(val_metrics["f1"])
        current_f1 = val_metrics["f1"]
    else:
        current_f1 = train_metrics["f1"]
        history["val_loss"].append(0)
        history["val_f1"].append(0)

    elapsed = time.time() - epoch_start

    val_loss_str = f'{val_metrics["loss"]:.4f}' if val_loader else "N/A"
    val_f1_str = f'{val_metrics["f1"]:.4f}' if val_loader else "N/A"
    print(f"{epoch:5d} | {train_metrics['loss']:10.4f} | {val_loss_str:>10} | "
          f"{train_metrics['f1']:9.4f} | {val_f1_str:>9} | {elapsed:5.1f}s")

    # Early stopping
    if current_f1 > best_val_f1:
        best_val_f1 = current_f1
        patience_counter = 0
        torch.save(model.state_dict(), "best_model.pt")
        print(f"       → New best model saved (F1={best_val_f1:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch} (patience={PATIENCE})")
            break

print(f"\n✓ Training complete. Best validation F1: {best_val_f1:.4f}")

# Section 9 — Evaluation
Load the best checkpoint and evaluate on the validation set.  
Reports: macro F1, per-class metrics, confusion matrix heatmap, and error analysis.

In [ ]:
# ========================== SECTION 9: EVALUATION ==========================

# Load best model
model.load_state_dict(torch.load("best_model.pt", map_location=device, weights_only=True))
model.eval()
print("✓ Loaded best model checkpoint")

if val_loader:
    val_results = eval_epoch(model, val_loader, criterion, device)
    y_true = val_results["labels"]
    y_pred = val_results["preds"]
else:
    # Evaluate on training set if no validation
    train_results = eval_epoch(model, train_loader, criterion, device)
    y_true = train_results["labels"]
    y_pred = train_results["preds"]
    print("(No validation set — evaluating on training set)")

# ---------- Metrics ----------
print(f"\n{'='*60}")
print(f"Evaluation Results — Subtask {SUBTASK}")
print(f"{'='*60}")
print(f"Macro F1  : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
print(f"Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
print(f"Precision : {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
print(f"Recall    : {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")

# Per-class report
class_names = [id_to_label.get(i, str(i)) for i in range(NUM_CLASSES)]
print(f"\nPer-class report:")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

# ---------- Confusion Matrix ----------
cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
fig, ax = plt.subplots(figsize=(max(6, NUM_CLASSES), max(5, NUM_CLASSES - 1)))
im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
ax.set_title(f"Confusion Matrix — Subtask {SUBTASK}", fontsize=14)
fig.colorbar(im, ax=ax, shrink=0.8)
tick_marks = list(range(NUM_CLASSES))
ax.set_xticks(tick_marks)
ax.set_xticklabels(class_names, rotation=45, ha="right")
ax.set_yticks(tick_marks)
ax.set_yticklabels(class_names)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")

# Annotate cells
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        color = "white" if cm[i, j] > cm.max() / 2 else "black"
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", color=color, fontsize=10)

plt.tight_layout()
plt.savefig(f"confusion_matrix_task_{SUBTASK}.png", dpi=150)
plt.show()
print(f"✓ Confusion matrix saved to confusion_matrix_task_{SUBTASK}.png")

In [ ]:
# ---------- Error Analysis ----------
print(f"\n{'='*60}")
print(f"Error Analysis — 5 Misclassified Examples Per Class")
print(f"{'='*60}")

eval_df = val_df if val_df is not None else train_df
eval_ds = val_dataset if val_dataset is not None else train_dataset

# Gather predictions with indices
errors_by_class = defaultdict(list)
for i, (true_label, pred_label) in enumerate(zip(y_true, y_pred)):
    if true_label != pred_label:
        errors_by_class[true_label].append((i, pred_label))

for cls_id in range(NUM_CLASSES):
    cls_name = id_to_label.get(cls_id, str(cls_id))
    errs = errors_by_class.get(cls_id, [])
    print(f"\n--- Class {cls_id} ({cls_name}): {len(errs)} errors ---")
    for idx, pred in errs[:5]:
        # Get original data index
        if idx < len(eval_ds.valid_indices):
            orig_idx = eval_ds.valid_indices[idx]
            if orig_idx < len(eval_df):
                code_snippet = eval_df.iloc[orig_idx]["code"][:150].replace("\n", " ")
                pred_name = id_to_label.get(pred, str(pred))
                print(f"  True={cls_name}, Pred={pred_name}: {code_snippet}...")

# Section 10 — Inference & Submission
Single-sample prediction function and batch submission generation.  
Exports to `submission_task_A.csv` (or B/C) with format validation.

In [ ]:
# ========================== SECTION 10: INFERENCE & SUBMISSION ==========================

def predict(model, code_str, language, parsers_dict, node_vocab_dict, dev):
    """Predict the label for a single code snippet.

    Args:
        model: Trained CodeGNN model.
        code_str: Source code string.
        language: Programming language string.
        parsers_dict: Dict of Tree-sitter parsers.
        node_vocab_dict: Node type vocabulary.
        dev: torch.device.

    Returns:
        Integer label ID.

    Raises:
        ValueError: If code cannot be parsed.
    """
    model.eval()
    root = parse_code(code_str, language, parsers_dict)
    if root is None:
        # Fallback: return most common class
        warnings.warn("Could not parse code — returning default label 0")
        return 0

    result = ast_to_graph(root, node_vocab_dict, MAX_NODES)
    if result is None:
        return 0

    nf, ei = result
    data = Data(x=nf, edge_index=ei, num_nodes=nf.shape[0])
    # Create a batch with a single graph
    batch_vec = torch.zeros(nf.shape[0], dtype=torch.long)
    data.batch = batch_vec
    data = data.to(dev)

    with torch.no_grad():
        logits = model(data.x, data.edge_index, data.batch)
        pred = logits.argmax(dim=1).item()
    return pred


def generate_submission(model, test_df, parsers_dict, node_vocab_dict, dev):
    """Generate submission CSV from test data.

    Args:
        model: Trained CodeGNN model.
        test_df: DataFrame with 'code' and 'language' columns.
        parsers_dict: Dict of Tree-sitter parsers.
        node_vocab_dict: Node type vocabulary.
        dev: torch.device.

    Returns:
        DataFrame with 'id' and 'label' columns.
    """
    model.eval()
    predictions = []

    for idx in tqdm(range(len(test_df)), desc="Generating predictions"):
        row = test_df.iloc[idx]
        lang = row.get("language", "python")
        pred = predict(model, row["code"], lang, parsers_dict, node_vocab_dict, dev)
        predictions.append({"id": idx, "label": int(pred)})

    sub_df = pd.DataFrame(predictions)
    return sub_df


# ---------- Demo: single prediction ----------
demo_row = train_df.iloc[0]
demo_pred = predict(model, demo_row["code"], demo_row.get("language", "python"),
                    parsers, node_vocab, device)
print(f"Demo prediction: true={demo_row['label']}, predicted={demo_pred}")

# ---------- Generate submission on validation set (as proxy for test) ----------
print(f"\nGenerating submission on {'validation' if val_df is not None else 'training'} set...")
sub_source = val_df if val_df is not None else train_df
submission = generate_submission(model, sub_source, parsers, node_vocab, device)

# ---------- Validate format ----------
assert list(submission.columns) == ["id", "label"], f"Unexpected columns: {submission.columns.tolist()}"
assert submission["label"].dtype in [np.int64, np.int32, int], "Label must be integer"
valid_label_ids = set(range(NUM_CLASSES))
invalid = set(submission["label"].unique()) - valid_label_ids
assert len(invalid) == 0, f"Invalid label IDs found: {invalid}"

# Save
out_path = f"submission_task_{SUBTASK}.csv"
submission.to_csv(out_path, index=False)

print(f"\n✓ Submission saved to {out_path}")
print(f"  Rows: {len(submission):,}")
print(f"  Label distribution:")
print(submission["label"].value_counts().to_string())
print(f"\nFirst 5 rows:")
print(submission.head())

### Training Curves

In [ ]:
# ---------- Plot training curves ----------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history["train_loss"], label="Train Loss", marker="o", markersize=3)
if any(v > 0 for v in history["val_loss"]):
    axes[0].plot(history["val_loss"], label="Val Loss", marker="s", markersize=3)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss Curves")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# F1
axes[1].plot(history["train_f1"], label="Train F1", marker="o", markersize=3)
if any(v > 0 for v in history["val_f1"]):
    axes[1].plot(history["val_f1"], label="Val F1", marker="s", markersize=3)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Macro F1")
axes[1].set_title("Macro F1 Curves")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"training_curves_task_{SUBTASK}.png", dpi=150)
plt.show()
print(f"✓ Training curves saved to training_curves_task_{SUBTASK}.png")